# Issue #7 — Crossover prediction

Walk-forward probability forecasts for 1-, 3-, and 5-day crossover horizons. Compare liquidity features against the calendar-only baseline. End-of-day signals are only tradable from the next session.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'src' / 'crossover_model.py').exists()), None)
if root is None:
    raise FileNotFoundError('Could not find src/crossover_model.py')
sys.path.insert(0, str(root))
from src.crossover_model import walk_forward_predictions, score_predictions
panel = pd.read_parquet(root / 'data/processed/zn_early_indicator_panel.parquet')


In [ ]:
tables = []
for horizon in (1, 3, 5):
    predictions = walk_forward_predictions(panel, horizon=horizon, min_train_rolls=6)
    scores = score_predictions(predictions)
    scores.insert(0, 'horizon', horizon)
    tables.append(scores)
    output = root / 'data/processed' / f'zn_crossover_predictions_{horizon}d.csv'
    predictions.to_csv(output, index=False)
results = pd.concat(tables, ignore_index=True)
display(results)


Lower Brier score means better probability forecasts. Compare both models on the same held-out rolls; do not treat the repeated daily observations as independent rolls.


In [ ]:
ax = results.pivot(index='horizon', columns='model', values='brier_roll_mean').plot.bar(figsize=(7, 4))
ax.set(xlabel='Prediction horizon (business days)', ylabel='Mean held-out-roll Brier score', title='Calendar vs. liquidity features')
plt.tight_layout()
plt.show()
